# 01 — Fetch: Sri Lankan mortality & longevity indicators

**Source:** [World Bank Open Data](https://data.worldbank.org/) API v2 (no key, no auth).

**Purpose of this notebook:** download 11 demographic indicators for Sri Lanka and 5 regional
comparators, and write them to `data/raw_worldbank.csv` **exactly as returned** — nulls included,
no filtering, no interpolation.

Fetching and cleaning are kept in separate notebooks on purpose. Every cleaning decision
(year window, missing values, dropped comparators) is then made *visibly* in `02_clean.ipynb`
against a raw file that can be re-inspected, rather than silently baked into the download.

## The API contract, verified before writing the loop

A single URL was opened first to confirm the response shape:

```
https://api.worldbank.org/v2/country/LKA/indicator/SP.DYN.LE00.IN?format=json&per_page=100
```

| Property | Observed |
|---|---|
| Top level | 2-element list: `[metadata, data]` |
| Rows per series | 66 — one per year, 1960 → 2025, newest first |
| Fields used | `countryiso3code`, `country.value`, `date`, `value` |
| Pagination | `total: 66` per series, so `per_page=500` never paginates |
| Data vintage | `lastupdated: 2026-07-13` |

**Coverage caveat found during verification:** the series extend to 2025 but the recent tail is
`null`. Life expectancy and the mortality rates carry real values to **2024**; the population
measures reach **2025**. This is measured explicitly at the end of this notebook and drives the
year-window decision in `02_clean.ipynb`.

In [1]:
import time
from pathlib import Path

import pandas as pd
import requests

# Indicator code -> the short column name used everywhere downstream.
INDICATORS = {
    "SP.DYN.LE00.IN":     "life_exp_total",
    "SP.DYN.LE00.MA.IN":  "life_exp_male",
    "SP.DYN.LE00.FE.IN":  "life_exp_female",
    "SP.DYN.AMRT.MA":     "adult_mortality_male",
    "SP.DYN.AMRT.FE":     "adult_mortality_female",
    "SP.DYN.IMRT.IN":     "infant_mortality",
    "SH.DYN.MORT":        "under5_mortality",
    "SP.DYN.CDRT.IN":     "crude_death_rate",
    "SP.POP.65UP.TO.ZS":  "pop_65plus_pct",
    "SP.POP.DPND.OL":     "old_age_dependency",
    "SP.POP.TOTL":        "population_total",
}

# Sri Lanka plus five South Asian comparators.
COUNTRIES = ["LKA", "IND", "BGD", "MDV", "NPL", "PAK"]

print(f"{len(COUNTRIES)} countries x {len(INDICATORS)} indicators = "
      f"{len(COUNTRIES) * len(INDICATORS)} requests")

6 countries x 11 indicators = 66 requests


## One request → one tidy frame

`fetch()` returns **long/tidy** rows (`country_code, country, year, indicator, value`) rather than
a wide table. Long format concatenates safely across 66 heterogeneous responses; the pivot to wide
happens once, later, in `02_clean.ipynb`.

The retry exists because a single dropped connection out of 66 would otherwise discard the whole
run. `value` is left untouched — a `null` from the API stays `NaN` here and is dealt with in
cleaning.

In [2]:
BASE = "https://api.worldbank.org/v2"


def fetch(country: str, code: str, retries: int = 3) -> pd.DataFrame:
    """Return tidy rows for one country x one indicator. Empty frame if no data."""
    url = f"{BASE}/country/{country}/indicator/{code}?format=json&per_page=500"

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            payload = r.json()
            break
        except (requests.RequestException, ValueError) as err:
            if attempt == retries:
                raise
            print(f"  retry {attempt}/{retries - 1} for {country}/{code}: {err}")
            time.sleep(2 * attempt)

    # A valid response is [metadata, rows]; element 1 is None when the series is empty.
    if len(payload) < 2 or payload[1] is None:
        print(f"  no data returned for {country}/{code}")
        return pd.DataFrame()

    return pd.DataFrame([
        {
            "country_code": row["countryiso3code"],
            "country": row["country"]["value"],
            "year": int(row["date"]),
            "indicator": INDICATORS[code],
            "value": row["value"],
        }
        for row in payload[1]
    ])

In [3]:
frames = []

for country in COUNTRIES:
    got = 0
    for code in INDICATORS:
        df = fetch(country, code)
        frames.append(df)
        got += len(df)
        time.sleep(0.2)          # be polite to a free public API
    print(f"{country}: {got:,} rows")

raw = pd.concat(frames, ignore_index=True)
print(f"\ncombined: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

LKA: 726 rows


IND: 726 rows


BGD: 726 rows


MDV: 726 rows


NPL: 726 rows


PAK: 726 rows

combined: 4,356 rows x 5 columns


## Sanity checks

Three questions before trusting the file: did every indicator arrive, is the grain what I think it
is (one row per country-year-indicator), and how much of it is actually populated?

In [4]:
print(f"indicators : {raw['indicator'].nunique()} of {len(INDICATORS)} expected")
print(f"countries  : {raw['country_code'].nunique()} of {len(COUNTRIES)} expected")
print(f"year range : {raw['year'].min()} - {raw['year'].max()}")

dupes = raw.duplicated(["country_code", "year", "indicator"]).sum()
print(f"duplicate country-year-indicator rows : {dupes}")

nulls = raw["value"].isna().sum()
print(f"null values : {nulls:,} of {len(raw):,} ({nulls / len(raw) * 100:.1f}%)")

raw.head()

indicators : 11 of 11 expected
countries  : 6 of 6 expected
year range : 1960 - 2025
duplicate country-year-indicator rows : 0
null values : 52 of 4,356 (1.2%)


,country_code,country,year,indicator,value
0,LKA,Sri Lanka,2025,life_exp_total,NaN
1,LKA,Sri Lanka,2024,life_exp_total,77.672
2,LKA,Sri Lanka,2023,life_exp_total,77.483
3,LKA,Sri Lanka,2022,life_exp_total,77.300
4,LKA,Sri Lanka,2021,life_exp_total,76.278


In [5]:
# Where does each series actually end? This is the input to the year-window decision in 02_clean.
coverage = (raw.dropna(subset=["value"])
               .groupby(["indicator", "country_code"])["year"]
               .agg(first_year="min", last_year="max", n_years="count"))

last_year_by_indicator = coverage.reset_index().pivot(
    index="indicator", columns="country_code", values="last_year"
)
print("Last year with data, by indicator and country:")
last_year_by_indicator

Last year with data, by indicator and country:


country_code,BGD,IND,LKA,MDV,NPL,PAK
indicator,,,,,,
adult_mortality_female,2024,2024,2024,2024,2024,2024
adult_mortality_male,2024,2024,2024,2024,2024,2024
crude_death_rate,2024,2024,2024,2024,2024,2024
infant_mortality,2024,2024,2024,2024,2024,2024
life_exp_female,2024,2024,2024,2024,2024,2024
life_exp_male,2024,2024,2024,2024,2024,2024
life_exp_total,2024,2024,2024,2024,2024,2024
old_age_dependency,2025,2025,2025,2025,2025,2025
pop_65plus_pct,2025,2025,2025,2025,2025,2025


In [6]:
# Which country-indicator series are patchiest? Candidates for exclusion in 02_clean.
sparse = coverage.sort_values("n_years").head(10)
print("Thinnest series (fewest years with data):")
sparse

Thinnest series (fewest years with data):


first_year  last_year  n_years
indicator              country_code                                
infant_mortality       MDV                 1962       2024       63
under5_mortality       MDV                 1962       2024       63
adult_mortality_female LKA                 1960       2024       65
                       BGD                 1960       2024       65
                       NPL                 1960       2024       65
                       PAK                 1960       2024       65
adult_mortality_male   BGD                 1960       2024       65
adult_mortality_female IND                 1960       2024       65
adult_mortality_male   LKA                 1960       2024       65
                       MDV                 1960       2024       65

In [7]:
out = Path("../data/raw_worldbank.csv")
out.parent.mkdir(exist_ok=True)
raw.to_csv(out, index=False)

print(f"wrote {out.resolve()}")
print(f"{raw.shape[0]:,} rows, {out.stat().st_size / 1024:.0f} KB")

wrote D:\F\Projects\sl-mortality-analysis\data\raw_worldbank.csv
4,356 rows, 190 KB


## Checkpoint

`data/raw_worldbank.csv` now holds every observation the World Bank returns for 11 indicators ×
6 countries, unmodified.

**Carried forward into `02_clean.ipynb`:**

- Life expectancy and mortality rates end at **2024**; population measures reach **2025**. A
  window of **1960–2024** keeps every column of the panel populated over the same years.
- Nulls are concentrated in the recent tail (2025) rather than scattered mid-series, so the
  missingness is a *coverage* problem, not a data-quality one — that distinction changes the
  correct remedy from interpolation to truncation.
- The thinnest series above are the candidates for dropping from the regional comparison chart.